# Daily norms calculations

### Imports

In [1]:
%load_ext autoreload
%autoreload 2

In [29]:
import polars as pl
from datetime import date
from statsmodels.nonparametric.smoothers_lowess import lowess
import numpy as np
import datetime

import climato.config as config
from climato.sources.meteo_france import daily
from climato.utils.datetime import utc_now_iso

In [3]:
DATA_DIR = config.get_data_dir()
RAW_DATA_DIR = config.get_raw_data_dir()
MANIFEST_DIR = DATA_DIR / "metadata" / "manifests"
MANIFEST_PATH = MANIFEST_DIR / "daily_files.parquet"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DAILY_DIR = PROCESSED_DIR / daily.MF_DAILY_RELATIVE_PATH

### Parameter norms

In [4]:
REF_START = date(1991, 1, 1)
REF_END = date(2020, 12, 31)

In [59]:
from pathlib import Path


station_ids = list(
    pl.read_csv(
        Path(config.PROJECT_ROOT / "dump" / "stations_candidates_20261018T183248Z.csv"),
        separator=";",
        schema_overrides={
            "station_id": pl.String,
        },
    )
    .select("station_id")
    .sort("station_id")
    .to_series()
)

print(station_ids)

departments = sorted({station_id[:2] for station_id in station_ids})

['01089001', '01235001', '02320001', '03059001', '03060001', '03100001', '03108001', '03130002', '03204001', '03240001', '03310001', '03321001', '04039001', '04049001', '05046001', '05142001', '06029001', '06088001', '07010001', '07019005', '07064001', '07066001', '07068001', '07075001', '07105001', '07117002', '07128001', '07159001', '07168001', '07261001', '07265001', '07331001', '07338001', '08419001', '09181003', '09289001', '10030001', '10057001', '10130001', '10323001', '11004001', '11012001', '11069001', '11076001', '11206003', '12145001', '12253001', '12254001', '13001009', '13004003', '13047001', '13054001', '13103001', '14137001', '14239001', '14578001', '14624001', '15014004', '15050001', '15118002', '15119001', '16089001', '17218001', '17224001', '17243002', '17268002', '17323001', '17353001', '17415003', '18033001', '18057003', '18092001', '18242001', '18279001', '19125001', '19136002', '19272001', '19275001', '20004002', '20041001', '20050001', '20148001', '20247001', '20

In [6]:
station_ids = (
    '06088001', '13054001', '14137001', '16089001', '20148001', '21473001', '25056001', '26198001',
    '29075001', '30189001', '31069001', '33281001', '35281001', '36063001', '44020001', '45055001',
    '47091001', '51183001', '51449002', '54526001',
    '58160001', '59343001', '63113001', '64549001', '66136001', '67124001', '69029001', '72181001',
    '73054001', '75114001', '86027001'
)

departments = sorted({station_id[:2] for station_id in station_ids})

In [60]:
params = ["tn", "tx", "tntxm", "tampli"]

is_ref = pl.col("obs_date").is_between(REF_START, REF_END)

df_obs = (
    pl.scan_parquet(
        PROCESSED_DAILY_DIR / "parameter_family=rr_t_vent",
        hive_partitioning=True,
        hive_schema={
            "dataset_year": pl.UInt16,
            "department": pl.String,
            "collection": pl.String,
            "period": pl.String,
        },
    )
    .filter(
        pl.col("collection") == "quot",
        pl.col("department").is_in(departments),
        pl.col("station_id").is_in(station_ids),
        # pl.col("obs_date").is_between(REF_START, REF_END),
    )
    .select(
        "station_id",
        "station_name",
        "obs_date",
        *(pl.col(param) for param in params),
    )
    .sort(
        "station_id",
        "obs_date",
    )
    .with_columns(
        pl.col("obs_date").dt.year().alias("year"),
        pl.col("obs_date").dt.month().alias("month"),
        pl.col("obs_date").dt.day().alias("day"),
        pl.col("obs_date").dt.strftime(format="%m-%d").alias("clim_day"),
    )
    # .with_columns(
    #     pl.date(year=pl.lit(2000), month=pl.col("month"), day=pl.col("day")).alias("plot_date"),
    # )
    # .explain(optimized=True)
    .select(
        "station_id", "station_name", "obs_date", "year", "month", "day", "clim_day",
        *(pl.col(param) for param in params),
    )
    .collect(engine="streaming")
)

with pl.Config(tbl_cols=-1, tbl_rows=100, tbl_width_chars=200):
    print(df_obs)

shape: (9_966_886, 11)
┌────────────┬──────────────┬────────────┬──────┬───────┬─────┬──────────┬──────┬──────┬───────┬────────┐
│ station_id ┆ station_name ┆ obs_date   ┆ year ┆ month ┆ day ┆ clim_day ┆ tn   ┆ tx   ┆ tntxm ┆ tampli │
│ ---        ┆ ---          ┆ ---        ┆ ---  ┆ ---   ┆ --- ┆ ---      ┆ ---  ┆ ---  ┆ ---   ┆ ---    │
│ str        ┆ str          ┆ date       ┆ i32  ┆ i8    ┆ i8  ┆ str      ┆ f64  ┆ f64  ┆ f64   ┆ f64    │
╞════════════╪══════════════╪════════════╪══════╪═══════╪═════╪══════════╪══════╪══════╪═══════╪════════╡
│ 01089001   ┆ AMBERIEU     ┆ 1934-03-01 ┆ 1934 ┆ 3     ┆ 1   ┆ 03-01    ┆ null ┆ null ┆ null  ┆ null   │
│ 01089001   ┆ AMBERIEU     ┆ 1934-03-02 ┆ 1934 ┆ 3     ┆ 2   ┆ 03-02    ┆ null ┆ null ┆ null  ┆ null   │
│ 01089001   ┆ AMBERIEU     ┆ 1934-03-03 ┆ 1934 ┆ 3     ┆ 3   ┆ 03-03    ┆ null ┆ null ┆ null  ┆ null   │
│ 01089001   ┆ AMBERIEU     ┆ 1934-03-04 ┆ 1934 ┆ 3     ┆ 4   ┆ 03-04    ┆ null ┆ null ┆ null  ┆ null   │
│ 01089001   ┆ AMBERIEU

In [15]:
stations_dfs = df_obs.partition_by(
    "station_id",
    as_dict=True,
)

In [16]:
clim_days = (
    pl.DataFrame(
        pl.date_range(date(2000, 1, 1), date(2000, 12, 31), eager=True).alias("date")
    )
    .with_row_index("clim_day_idx")
    .with_columns(
        clim_day=pl.col("date").dt.strftime(format="%m-%d"),
        month=pl.col("date").dt.month(),
        day=pl.col("date").dt.day(),
    )
)

print(clim_days)

shape: (366, 5)
┌──────────────┬────────────┬──────────┬───────┬─────┐
│ clim_day_idx ┆ date       ┆ clim_day ┆ month ┆ day │
│ ---          ┆ ---        ┆ ---      ┆ ---   ┆ --- │
│ u32          ┆ date       ┆ str      ┆ i8    ┆ i8  │
╞══════════════╪════════════╪══════════╪═══════╪═════╡
│ 0            ┆ 2000-01-01 ┆ 01-01    ┆ 1     ┆ 1   │
│ 1            ┆ 2000-01-02 ┆ 01-02    ┆ 1     ┆ 2   │
│ 2            ┆ 2000-01-03 ┆ 01-03    ┆ 1     ┆ 3   │
│ 3            ┆ 2000-01-04 ┆ 01-04    ┆ 1     ┆ 4   │
│ 4            ┆ 2000-01-05 ┆ 01-05    ┆ 1     ┆ 5   │
│ …            ┆ …          ┆ …        ┆ …     ┆ …   │
│ 361          ┆ 2000-12-27 ┆ 12-27    ┆ 12    ┆ 27  │
│ 362          ┆ 2000-12-28 ┆ 12-28    ┆ 12    ┆ 28  │
│ 363          ┆ 2000-12-29 ┆ 12-29    ┆ 12    ┆ 29  │
│ 364          ┆ 2000-12-30 ┆ 12-30    ┆ 12    ┆ 30  │
│ 365          ┆ 2000-12-31 ┆ 12-31    ┆ 12    ┆ 31  │
└──────────────┴────────────┴──────────┴───────┴─────┘


In [ ]:
def circular_lowess_span(values, span_days=31, it=0):
    y = np.asarray(values, dtype=float)
    n = len(y)
    x = np.arange(n, dtype=float)

    valid_original = np.isfinite(y)
    n_valid = valid_original.sum()

    if n_valid < span_days:
        raise ValueError(
            f"Not enough valid climatological days: "
            f"n_valid={n_valid} < span_days={span_days}"
        )

    x_ext = np.concatenate([x - n, x, x + n])
    y_ext = np.concatenate([y, y, y])

    valid_ext = np.isfinite(y_ext)

    x_fit = x_ext[valid_ext]
    y_fit = y_ext[valid_ext]

    if not np.isfinite(x_fit).all():
        raise ValueError("Non-finite x_fit")
    if not np.isfinite(y_fit).all():
        raise ValueError("Non-finite y_fit")
    if not np.isfinite(x_ext).all():
        raise ValueError("Non-finite xvals/x_ext")
    
    frac = min(span_days / len(y_fit), 1.0)

    smooth_ext = lowess(
        endog=y_fit,
        exog=x_fit,
        xvals=x_ext,
        frac=frac,
        it=it,
        is_sorted=True,
        # return_sorted=False, # ignored if xvals provided
        missing="none",
    )

    return smooth_ext[n:2*n]


def log_var_lowess_span(y, span_days=31, it=0):
    var = y ** 2
    epsilon = 1e-6
    log_var = np.log(np.maximum(var, epsilon))
    log_var_smooth = circular_lowess_span(log_var, span_days=span_days, it=it)
    std_smooth = np.sqrt(np.exp(log_var_smooth))

    return std_smooth


def compute_df_daily_normals(
        df_obs: pl.DataFrame,
        ref_start: datetime.date = REF_START,
        ref_end: datetime.date = REF_END,
):
    expressions = []
    for param in params:
        expressions.extend([
            pl.col(param).count().alias(f"{param}_n_obs"),
            pl.col(param).mean().alias(f"{param}_mean_raw"),
            pl.col(param).std().alias(f"{param}_std_raw"),
            pl.col(param).min().alias(f"{param}_min_raw"),
            pl.col(param).quantile(0.1).alias(f"{param}_q10_raw"),
            pl.col(param).quantile(0.25).alias(f"{param}_q25_raw"),
            pl.col(param).median().alias(f"{param}_median_raw"),
            pl.col(param).quantile(0.75).alias(f"{param}_q75_raw"),
            pl.col(param).quantile(0.9).alias(f"{param}_q90_raw"),
            pl.col(param).max().alias(f"{param}_max_raw"),
        ])

    df_daily_normals = (
        df_obs
        .filter(pl.col("obs_date").is_between(ref_start, ref_end))
        .group_by(["station_id", "station_name", "month", "day", "clim_day"])
        .agg(expressions)
    )

    df_daily_normals = (
        clim_days
        .join(
            df_daily_normals,
            how="left",
            on=["month", "day", "clim_day"],
        )
        .sort(["station_id", "month", "day"])
    )

    to_concat_smoothed = {col: [] for param in params for col in (f"{param}_mean_smooth", f"{param}_std_smooth")}
    for station_id in station_ids:
        station_normals = df_daily_normals.filter(pl.col("station_id") == station_id)
        for param in params:
            to_concat_smoothed[f"{param}_mean_smooth"].extend(
                circular_lowess_span(station_normals[f"{param}_mean_raw"].to_numpy(), span_days=45).tolist()
            )
            to_concat_smoothed[f"{param}_std_smooth"].extend(
                log_var_lowess_span(station_normals[f"{param}_std_raw"].to_numpy(), span_days=45).tolist()
            )

    for param in params:
        df_daily_normals = df_daily_normals.with_columns(
            pl.Series(
                f"{param}_mean_smooth", 
                to_concat_smoothed[f"{param}_mean_smooth"],
            ),
            pl.Series(
                f"{param}_std_smooth",
                to_concat_smoothed[f"{param}_std_smooth"],
            ),
        )

    columns_select = ["station_id", "station_name", "month", "day", "clim_day"]
    for param in params:
        columns_select.extend([
            pl.col(f"{param}_n_obs"),
            pl.col(f"{param}_mean_raw"),
            pl.col(f"{param}_std_raw"),
            pl.col(f"{param}_mean_smooth"),
            pl.col(f"{param}_std_smooth"),
            pl.col(f"{param}_min_raw"),
            pl.col(f"{param}_q10_raw"),
            pl.col(f"{param}_q25_raw"),
            pl.col(f"{param}_median_raw"),
            pl.col(f"{param}_q75_raw"),
            pl.col(f"{param}_q90_raw"),
            pl.col(f"{param}_max_raw"),
        ])

    return (
        df_daily_normals
        .select(
            *columns_select,
        )
    )


def compute_df_monthly_normals(
        df_obs: pl.DataFrame,
        ref_start: datetime.date = REF_START,
        ref_end: datetime.date = REF_END,
):
    months_ref_expressions = []
    for param in params:
        months_ref_expressions.append(
            pl.col(param).mean().alias(f"{param}m")
        )

        if param in ["tn", "tx"]:
            months_ref_expressions.append((pl.col(param) >= 25).sum().alias(f"{param}_nb_days_gt_25"))


    df_months_ref = (
        df_obs
        .filter(pl.col("obs_date").is_between(ref_start, ref_end))
        .group_by("station_id", "station_name", "year", "month")
        .agg(
            months_ref_expressions
        )
    )

    normals_expressions = []
    for param in params:
        m_col = f"{param}m"
        normals_expressions.extend([
            pl.col(m_col).mean().alias(f"{param}m_mean"),
            pl.col(m_col).std().alias(f"{param}m_std"),
            pl.col(m_col).median().alias(f"{param}m_median"),
            pl.col(m_col).quantile(0.1).alias(f"{param}m_q10"),
            pl.col(m_col).quantile(0.25).alias(f"{param}m_q25"),
            pl.col(m_col).quantile(0.75).alias(f"{param}m_q75"),
            pl.col(m_col).quantile(0.9).alias(f"{param}m_q90"),
        ])

    df_monthly_normals = (
        df_months_ref
        .group_by(["station_id", "station_name", "month"])
        .agg(normals_expressions)
        .sort("station_id", "month")
    )

    return df_monthly_normals

In [ ]:
df_norm_nice = compute_df_daily_normals(df_obs.filter(pl.col("station_id") == "06088001"))

In [35]:
print(df_norm_nice)
ts = utc_now_iso()
df_norm_nice.write_csv(config.PROJECT_ROOT / "dump" / f"daily_norms_nice_06088001_{ts}.csv", separator=";", decimal_comma=True)

shape: (366, 45)
┌────────────┬─────────────┬───────┬─────┬───┬─────────────┬─────────────┬────────────┬────────────┐
│ station_id ┆ station_nam ┆ month ┆ day ┆ … ┆ tampli_q10_ ┆ tampli_q25_ ┆ tampli_q75 ┆ tampli_q90 │
│ ---        ┆ e           ┆ ---   ┆ --- ┆   ┆ raw         ┆ raw         ┆ _raw       ┆ _raw       │
│ str        ┆ ---         ┆ i8    ┆ i8  ┆   ┆ ---         ┆ ---         ┆ ---        ┆ ---        │
│            ┆ str         ┆       ┆     ┆   ┆ f64         ┆ f64         ┆ f64        ┆ f64        │
╞════════════╪═════════════╪═══════╪═════╪═══╪═════════════╪═════════════╪════════════╪════════════╡
│ 06088001   ┆ NICE        ┆ 1     ┆ 1   ┆ … ┆ 4.7         ┆ 6.0         ┆ 9.4        ┆ 10.1       │
│ 06088001   ┆ NICE        ┆ 1     ┆ 2   ┆ … ┆ 3.3         ┆ 4.2         ┆ 8.8        ┆ 10.5       │
│ 06088001   ┆ NICE        ┆ 1     ┆ 3   ┆ … ┆ 6.0         ┆ 7.4         ┆ 9.9        ┆ 10.3       │
│ 06088001   ┆ NICE        ┆ 1     ┆ 4   ┆ … ┆ 3.0         ┆ 4.5         ┆

In [42]:
df_monthly_normals_nice = compute_df_monthly_normals(df_obs.filter(pl.col("station_id") == "06088001"))


In [44]:
print(df_monthly_normals_nice)
ts = utc_now_iso()
df_monthly_normals_nice.write_csv(config.PROJECT_ROOT / "dump" / f"monthly_normals_nice_06088001_{ts}.csv", separator=";", decimal_comma=True)

shape: (12, 31)
┌────────────┬────────────┬───────┬───────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ station_id ┆ station_na ┆ month ┆ tnm_mean  ┆ … ┆ tamplim_q1 ┆ tamplim_q ┆ tamplim_q ┆ tamplim_q │
│ ---        ┆ me         ┆ ---   ┆ ---       ┆   ┆ 0          ┆ 25        ┆ 75        ┆ 90        │
│ str        ┆ ---        ┆ i8    ┆ f64       ┆   ┆ ---        ┆ ---       ┆ ---       ┆ ---       │
│            ┆ str        ┆       ┆           ┆   ┆ f64        ┆ f64       ┆ f64       ┆ f64       │
╞════════════╪════════════╪═══════╪═══════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 06088001   ┆ NICE       ┆ 1     ┆ 5.774194  ┆ … ┆ 6.277419   ┆ 6.541935  ┆ 8.380645  ┆ 8.741935  │
│ 06088001   ┆ NICE       ┆ 2     ┆ 6.069175  ┆ … ┆ 6.428571   ┆ 6.835714  ┆ 8.042857  ┆ 8.258621  │
│ 06088001   ┆ NICE       ┆ 3     ┆ 8.256344  ┆ … ┆ 6.119355   ┆ 6.719355  ┆ 7.719355  ┆ 8.1       │
│ 06088001   ┆ NICE       ┆ 4     ┆ 10.769111 ┆ … ┆ 5.83       ┆ 6.063333  

In [62]:
df_daily_normals = compute_df_daily_normals(df_obs)

In [56]:
print(df_daily_normals.height)

11346


In [ ]:
ts = utc_now_iso()
df_daily_normals.write_csv(config.PROJECT_ROOT / "dump" / f"daily_normals_333_{ts}.csv", separator=";", decimal_comma=True)

In [64]:
columns_select_normals_before = ["station_id", "clim_day"]
for param in params:
    columns_select_normals_before.extend([
        pl.col(f"{param}_mean_smooth"),
        pl.col(f"{param}_std_smooth"),
    ])

columns_select_obs_after = ["station_id", "station_name", "obs_date", "clim_day", "year", "month", "day"]
for param in params:
    columns_select_obs_after.extend([
        pl.col(param),
        pl.col(f"z_{param}"),
        pl.col(f"{param}_mean_smooth"),
        pl.col(f"{param}_std_smooth"),
        pl.col(f"ano_{param}"),
    ])

expressions = []
for param in params:
    expressions.append((pl.col(param) - pl.col(f"{param}_mean_smooth")).alias(f"ano_{param}"))
    expressions.append(((pl.col(param) - pl.col(f"{param}_mean_smooth")) / pl.col(f"{param}_std_smooth")).alias(f"z_{param}"))

df_obs_w_normals = (
    df_obs
    .join(
        df_daily_normals.select(columns_select_normals_before),
        how="inner",
        on=["station_id", "clim_day"]
    )
    .with_columns(expressions)
    .select(columns_select_obs_after)
    .sort("station_id", "obs_date")
)

In [ ]:
head_obs = (
    df_obs_w_normals
    .lazy()
    .drop_nulls(["z_tn", "z_tx"])
    .filter(
        # pl.col("month") == 12,
        # pl.col("station_id") == "54526001",
        pl.col("station_id").str.starts_with("13"),
    )
    .with_columns((pl.col("z_tx") - pl.col("z_tn")).alias("z_tx_tn"))
    .sort(pl.col("z_tn"), descending=False)
    .head(100)
    .collect()
)

with pl.Config(tbl_rows=100, tbl_cols=50, tbl_width_chars=500):
    print(head_obs)

In [90]:
head_obs = (
    df_obs_w_normals
    .lazy()
    .drop_nulls(["z_tn", "z_tx"])
    .filter(
        # pl.col("month") == 12,
        # pl.col("station_id") == "54526001",
        pl.col("station_id").str.starts_with("83"),
    )
    .with_columns((pl.col("z_tx") - pl.col("z_tn")).alias("z_tx_tn"))
    .sort(pl.col("z_tx"), descending=False)
    .head(100)
    .collect()
)

with pl.Config(tbl_rows=100, tbl_cols=50, tbl_width_chars=500):
    print(head_obs)

shape: (100, 28)
┌────────────┬────────────────────┬────────────┬──────────┬──────┬───────┬─────┬───────┬───────────┬────────────────┬───────────────┬────────────┬──────┬───────────┬────────────────┬───────────────┬────────────┬───────┬───────────┬───────────────────┬──────────────────┬────────────┬────────┬───────────┬────────────────────┬───────────────────┬────────────┬───────────┐
│ station_id ┆ station_name       ┆ obs_date   ┆ clim_day ┆ year ┆ month ┆ day ┆ tn    ┆ z_tn      ┆ tn_mean_smooth ┆ tn_std_smooth ┆ ano_tn     ┆ tx   ┆ z_tx      ┆ tx_mean_smooth ┆ tx_std_smooth ┆ ano_tx     ┆ tntxm ┆ z_tntxm   ┆ tntxm_mean_smooth ┆ tntxm_std_smooth ┆ ano_tntxm  ┆ tampli ┆ z_tampli  ┆ tampli_mean_smooth ┆ tampli_std_smooth ┆ ano_tampli ┆ z_tx_tn   │
│ ---        ┆ ---                ┆ ---        ┆ ---      ┆ ---  ┆ ---   ┆ --- ┆ ---   ┆ ---       ┆ ---            ┆ ---           ┆ ---        ┆ ---  ┆ ---       ┆ ---            ┆ ---           ┆ ---        ┆ ---   ┆ ---       ┆ ---      

In [69]:
head_obs_tx = df_obs_w_normals.lazy().drop_nulls("tx").sort("tx", descending=True).head(100).collect()
with pl.Config(tbl_rows=100, tbl_cols=50, tbl_width_chars=500):
    print(head_obs_tx)

shape: (100, 27)
┌────────────┬─────────────────────────────┬────────────┬──────────┬──────┬───────┬─────┬──────┬───────────┬────────────────┬───────────────┬───────────┬──────┬──────────┬────────────────┬───────────────┬───────────┬───────┬──────────┬───────────────────┬──────────────────┬───────────┬────────┬──────────┬────────────────────┬───────────────────┬────────────┐
│ station_id ┆ station_name                ┆ obs_date   ┆ clim_day ┆ year ┆ month ┆ day ┆ tn   ┆ z_tn      ┆ tn_mean_smooth ┆ tn_std_smooth ┆ ano_tn    ┆ tx   ┆ z_tx     ┆ tx_mean_smooth ┆ tx_std_smooth ┆ ano_tx    ┆ tntxm ┆ z_tntxm  ┆ tntxm_mean_smooth ┆ tntxm_std_smooth ┆ ano_tntxm ┆ tampli ┆ z_tampli ┆ tampli_mean_smooth ┆ tampli_std_smooth ┆ ano_tampli │
│ ---        ┆ ---                         ┆ ---        ┆ ---      ┆ ---  ┆ ---   ┆ --- ┆ ---  ┆ ---       ┆ ---            ┆ ---           ┆ ---       ┆ ---  ┆ ---      ┆ ---            ┆ ---           ┆ ---       ┆ ---   ┆ ---      ┆ ---               ┆ ---  